# ❓ 1. Research Question (研究問題)

Before conducting the inferential analysis, we define our core statistical questions:

* **Gender and Alcohol Association (性別與飲酒關聯性):** "Is there a statistically significant association between a student's biological sex and their current alcohol use status?" (Using Chi-Square Test of Independence, $\text{df}=1$)
* **Age and Alcohol Association (年齡與飲酒關聯性):** "Does the proportion of current alcohol use change significantly across different adolescent ages (12–18)?" (Using Chi-Square Test of Independence, $\text{df}=6$)

---

# 📊 2. Variable Definition and Data Check (變數定義與資料檢查)

In this section, we define the variables used and document the data cleaning process to ensure reproducibility:

### 2.1 Grouping Variable: `WhatIsYourSex` $\rightarrow$ `Sex_Binary`
* **Definition:** Biological sex of the high school student.
* **Recoding Rules:** * `0` = Male (Originally coded as 2)
    * `1` = Female (Originally coded as 1)

### 2.2 Response Variable: `CurrentAlcoholUse` $\rightarrow$ `Alcohol_Binary`
* **Definition:** Measures whether students have consumed alcohol currently (within the past 30 days).
* **Recoding Rules:** * `0` = No current use (Originally coded as 1)
    * `1` = Active user (Originally coded as 2 to 7)

### 2.3 New Demographic Variable: `HowOldAreYou` $\rightarrow$ `Age_Numeric`
* **Definition:** Chronological age of the student (Discrete integer, focused on ages 12–18).

### 2.4 Final Sample Size
* **Data Cleaning:** Handled using explicit missing-value dropping and row parsing.
* **Final Analysis Sample Size:** $N = 12,615$

In [3]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import os

raw_filename = 'YRBS_2007.csv'

if os.path.exists(raw_filename):
    print(f"✅ 找到原始資料：{raw_filename}")
    raw_df = pd.read_csv(raw_filename)
    
    # 提取目標分析欄位
    target_vars = ['WhatIsYourSex', 'CurrentAlcoholUse', 'HowOldAreYou']
    df_analyzed = raw_df[target_vars].dropna().copy()
    
    # 執行重編碼
    df_analyzed['Sex_Binary'] = df_analyzed['WhatIsYourSex'].apply(lambda x: 1 if x == 1 else (0 if x == 2 else np.nan))
    df_analyzed['Alcohol_Binary'] = df_analyzed['CurrentAlcoholUse'].apply(lambda x: 0 if x == 1 else (1 if 2 <= x <= 7 else np.nan))
    
    # 將年齡代碼 1-7 轉換為實際整數年齡 12-18
    age_mapping = {1: 12, 2: 13, 3: 14, 4: 15, 5: 16, 6: 17, 7: 18}
    df_analyzed['Age_Numeric'] = df_analyzed['HowOldAreYou'].map(age_mapping)
    
    # 排除重編碼後產生的潛在缺失值並過濾標準年齡區間
    df_analyzed = df_analyzed.dropna(subset=['Sex_Binary', 'Alcohol_Binary', 'Age_Numeric']).copy()
    df_analyzed['Age_Numeric'] = df_analyzed['Age_Numeric'].astype(int)
    df_analyzed = df_analyzed[df_analyzed['Age_Numeric'].between(12, 18)]
    
    print(f"✅ 資料處理完成，最終有效分析樣本數：{len(df_analyzed)}")
else:
    print(f"❌ 錯誤：在目前的資料夾找不到 {raw_filename}")
    raise FileNotFoundError

✅ 找到原始資料：YRBS_2007.csv
✅ 資料處理完成，最終有效分析樣本數：12615


---

## 📊 3. Descriptive Summary Table (描述性統計摘要)


In [6]:
# 建立性別描述性統計摘要表
sex_summary = df_analyzed.groupby('Sex_Binary')['Alcohol_Binary'].agg(['count', 'sum', 'mean'])
sex_summary.columns = ['Sample Size (n)', 'Current Alcohol Users (count)', 'Proportion (p_hat)']
sex_summary.index = ['Male (0)', 'Female (1)']
print("=== 1. 性別與飲酒描述性統計 ===")
print(sex_summary)

=== 1. 性別與飲酒描述性統計 ===
            Sample Size (n)  Current Alcohol Users (count)  Proportion (p_hat)
Male (0)               6213                           2842            0.457428
Female (1)             6402                           2851            0.445330


In [8]:
# 建立年齡層描述性統計摘要表
age_summary = df_analyzed.groupby('Age_Numeric')['Alcohol_Binary'].agg(['count', 'sum', 'mean'])
age_summary.columns = ['Sample Size (n)', 'Current Alcohol Users (count)', 'Proportion (p_hat)']
print("\n=== 2. 年齡層與飲酒描述性統計 ===")
print(age_summary)


=== 2. 年齡層與飲酒描述性統計 ===
             Sample Size (n)  Current Alcohol Users (count)  \
Age_Numeric                                                   
12                        12                             10   
13                         6                              3   
14                      1253                            413   
15                      2889                           1132   
16                      3252                           1480   
17                      3245                           1614   
18                      1958                           1041   

             Proportion (p_hat)  
Age_Numeric                      
12                     0.833333  
13                     0.500000  
14                     0.329609  
15                     0.391831  
16                     0.455105  
17                     0.497381  
18                     0.531665  


# 📈 4. Multi-Variable Statistical Inference (多變數推論統計)

We conduct two independent **Chi-Square ($\chi^2$) Tests of Independence** to test our hypotheses.

### Test 1: Sex vs. Alcohol Use
* **$H_0$:** Biological sex and current alcohol use status are independent (no association).
* **$H_a$:** Biological sex and current alcohol use status are dependent (associated).

### Test 2: Age Group vs. Alcohol Use (新增)
* **$H_0$:** Chronological age and current alcohol use status are independent.
* **$H_a$:** Chronological age and current alcohol use status are dependent (alcohol consumption rates shift with age).

In [11]:
# 1. 執行性別與飲酒的卡方檢定 (2x2 表格, df=1)
sex_contingency = pd.crosstab(df_analyzed['Sex_Binary'], df_analyzed['Alcohol_Binary'])
chi2_sex, p_val_sex, dof_sex, expected_sex = stats.chi2_contingency(sex_contingency)

# 2. 執行年齡與飲酒的卡方檢定 (7x2 表格, df=6)
age_contingency = pd.crosstab(df_analyzed['Age_Numeric'], df_analyzed['Alcohol_Binary'])
chi2_age, p_val_age, dof_age, expected_age = stats.chi2_contingency(age_contingency)

print("=== 1. 性別檢定結果 ===")
print(f"Chi-Square Statistic: {chi2_sex:.4f}, df: {dof_sex}, P-value: {p_val_sex:.4f}")

print("\n=== 2. 年齡檢定結果 ===")
print(f"Chi-Square Statistic: {chi2_age:.4f}, df: {dof_age}, P-value: {p_val_age:.4e}")

=== 1. 性別檢定結果 ===
Chi-Square Statistic: 1.8152, df: 1, P-value: 0.1779

=== 2. 年齡檢定結果 ===
Chi-Square Statistic: 202.4067, df: 6, P-value: 5.8328e-41


# 📝 5. Statistical Interpretation

* **Sex Interpretation (Gender Association):** The current alcohol consumption rate is 45.8% for high school males and 44.6% for females. The Chi-Square Test of Independence yields $\chi^2 = 1.8152$ (with degrees of freedom $\text{df} = 1$) and a two-tailed $p\text{-value} = 0.1789$. Since $p > 0.05$, we **fail to reject the null hypothesis**. This indicates that there is no statistically significant association between biological sex and current alcohol use behavior among high school students as a whole.
* **Age Interpretation (Age Association):** Descriptive statistics demonstrate that alcohol consumption rates escalate dramatically as adolescents grow older (e.g., the alcohol use rate hovers around 30% at age 14, but surges close to the 50–60% range by age 18). The Chi-Square test for age yields an extremely small $p\text{-value}$ ($p < 0.001$), leading us to **strongly reject the null hypothesis**. This provides robust empirical evidence that there is a highly significant statistical association between chronological age and adolescent alcohol use, reflecting a clear developmental trajectory.

---

# 📋 6. Comprehensive Statistical Summary Table Export

We have restructured the summary table layout to simultaneously integrate both core independent variables (**Sex** and **Age**) into the final official reporting framework:

In [18]:
# =========================================================================
# 6. Construct and Export the Comprehensive Statistical Summary Table (Full English)
# =========================================================================

# Step 1: Compute the fine-grained breakdown by Age AND Sex for the updated table
age_sex_summary = df_analyzed.groupby(['Age_Numeric', 'Sex_Binary'])['Alcohol_Binary'].agg(['count', 'mean'])

# Step 2: Initialize lists to dynamically build the dictionary rows safely
variables = ["Sex_Binary (Overall)", "Sex_Binary (Overall)"]
data_types = ["Binary Categorical", "Binary Categorical"]
categories = ["Male (0)", "Female (1)"]
counts = [int(sex_summary.iloc[0]['Sample Size (n)']), int(sex_summary.iloc[1]['Sample Size (n)'])]
proportions = [f"{sex_summary.iloc[0]['Proportion (p_hat)']*100:.2f}%", f"{sex_summary.iloc[1]['Proportion (p_hat)']*100:.2f}%"]
methods = ["Chi-Square Test of Independence", "-"]
dfs = [int(dof_sex), "-"]
chi2s = [f"{chi2_sex:.4f}", "-"]
p_values = [f"{p_val_sex:.4f}", "-"]

# Step 3: Loop through ages 12 to 18 to dynamically append Total, Male, and Female rows
for idx, age in enumerate(range(12, 19)):
    # 3.1 Total for this specific age cohort
    variables.append(f"Age_Numeric (Age {age})")
    data_types.append("Discrete Numeric")
    categories.append(f"Age {age} - Total")
    counts.append(int(age_summary.loc[age, 'Sample Size (n)']))
    proportions.append(f"{age_summary.loc[age, 'Proportion (p_hat)']*100:.2f}%")
    # We display the multi-level Age Chi-Square test details only on the first age row as a reference
    methods.append("Chi-Square Test of Independence" if idx == 0 else "-")
    dfs.append(int(dof_age) if idx == 0 else "-")
    chi2s.append(f"{chi2_age:.4f}" if idx == 0 else "-")
    p_values.append("< 0.0001" if idx == 0 else "-")
    
    # 3.2 Male breakdown for this specific age
    variables.append(f"Age_Numeric (Age {age})")
    data_types.append("Discrete Numeric")
    categories.append(f"Age {age} - Male (0)")
    counts.append(int(age_sex_summary.loc[(age, 0), 'count']))
    proportions.append(f"{age_sex_summary.loc[(age, 0), 'mean']*100:.2f}%")
    methods.append("-")
    dfs.append("-")
    chi2s.append("-")
    p_values.append("-")
    
    # 3.3 Female breakdown for this specific age
    variables.append(f"Age_Numeric (Age {age})")
    data_types.append("Discrete Numeric")
    categories.append(f"Age {age} - Female (1)")
    counts.append(int(age_sex_summary.loc[(age, 1), 'count']))
    proportions.append(f"{age_sex_summary.loc[(age, 1), 'mean']*100:.2f}%")
    methods.append("-")
    dfs.append("-")
    chi2s.append("-")
    p_values.append("-")

# Step 4: Combine into the final professional structured dictionary
inference_summary_data = {
    "Variable": variables,
    "Data Type": data_types,
    "Analysis Group / Category": categories,
    "Sample Size (Count)": counts,
    "Alcohol Use Proportion (%)": proportions,
    "Statistical Method": methods,
    "Degrees of Freedom (df)": dfs,
    "Chi-Square Statistic (Chi2)": chi2s,
    "Asymptotic P-value": p_values
}

# Convert dictionary to DataFrame
df_summary_custom = pd.DataFrame(inference_summary_data)

# Define the export path
output_file = 'inference_summary_table.csv'

# Export to CSV with utf-8-sig to ensure universal Excel compatibility
df_summary_custom.to_csv(output_file, index=False, encoding='utf-8-sig')

print("=== Upgraded Comprehensive Statistical Inference Summary Table (Sex & Age Split) ===")
display(df_summary_custom)
print(f"\n💾 Success! The fine-grained full-English inference summary table has been saved to: {output_file}")

=== Upgraded Comprehensive Statistical Inference Summary Table (Sex & Age Split) ===


,Variable,Data Type,Analysis Group / Category,Sample Size (Count),Alcohol Use Proportion (%),Statistical Method,Degrees of Freedom (df),Chi-Square Statistic (Chi2),Asymptotic P-value
0,Sex_Binary (Overall),Binary Categorical,Male (0),6213,45.74%,Chi-Square Test of Independence,1,1.8152,0.1779
1,Sex_Binary (Overall),Binary Categorical,Female (1),6402,44.53%,-,-,-,-
2,Age_Numeric (Age 12),Discrete Numeric,Age 12 - Total,12,83.33%,Chi-Square Test of Independence,6,202.4067,< 0.0001
3,Age_Numeric (Age 12),Discrete Numeric,Age 12 - Male (0),6,83.33%,-,-,-,-
4,Age_Numeric (Age 12),Discrete Numeric,Age 12 - Female (1),6,83.33%,-,-,-,-
5,Age_Numeric (Age 13),Discrete Numeric,Age 13 - Total,6,50.00%,-,-,-,-
6,Age_Numeric (Age 13),Discrete Numeric,Age 13 - Male (0),3,33.33%,-,-,-,-
7,Age_Numeric (Age 13),Discrete Numeric,Age 13 - Female (1),3,66.67%,-,-,-,-
8,Age_Numeric (Age 14),Discrete Numeric,Age 14 - Total,1253,32.96%,-,-,-,-
9,Age_Numeric (Age 14),Discrete Numeric,Age 14 - Male (0),595,27.23%,-,-,-,-



💾 Success! The fine-grained full-English inference summary table has been saved to: inference_summary_table.csv
